# Bisection, False Position, and a Hybrid Method

Four independent equations, each solved with three closed root-finding methods: bisection, false
position, and a hybrid scheme that switches to bisection whenever the false-position secant slope
gets too steep. Each case reports iterations vs. target error (10^-2 down to 10^-12).

a)

$$
\sqrt{x} - \cos(x) = 0
$$

### Bisection, false position, and hybrid method

In [9]:
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np
import scipy as sp
import sympy as smp
from scipy.integrate import quad

In [ ]:


# Plot style: white background with a clearly visible grid (cosmetic only)
sns.set_style("whitegrid")
plt.rcParams["axes.facecolor"] = "white"
plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.edgecolor"] = "black"
plt.rcParams["grid.color"] = "black"
plt.rcParams["grid.linestyle"] = "-"
plt.rcParams["grid.linewidth"] = 1
plt.rcParams["xtick.color"] = "black"
plt.rcParams["ytick.color"] = "black"
plt.rcParams["axes.labelcolor"] = "black"
plt.rcParams["axes.titlecolor"] = "black"



In [ ]:
# Function under study
def f(x):
    return np.sqrt(x) - np.cos(x)

# General parameters
error_tol = 1e-6
max_iter = 1000

# Interval [a, b]
a, b = 0, 2

# Bisection method
def bisection(a, b, f, error_tol):
    valid_bracket = 0
    error = 1
    iterations = 0
    x_prev = 0

    # Validate the bracket
    if f(a) * f(b) < 0:
        valid_bracket = 1
    elif f(a) * f(b) == 0:
        if f(a) == 0:
            return a, 0, 0
        if f(b) == 0:
            return b, 0, 0
        return None, None, None

    estimated_iterations = int(np.log2((b - a) / error_tol))

    while valid_bracket == 1 and error > error_tol and iterations < max_iter:
        x_new = (a + b) / 2

        if f(a) * f(x_new) < 0:
            b = x_new
        else:
            a = x_new

        # Guard against division by zero
        if x_new != 0:
            error = abs((x_new - x_prev) / x_new)
        else:
            error = 0

        x_prev = x_new
        iterations += 1

        if f(x_new) == 0:
            break

    return x_new, error, iterations

# False position method
def false_position(a, b, f, error_tol):
    x_0 = 0
    error_act = 1
    iterations = 0

    if f(a) * f(b) > 0:
        raise ValueError("The function does not change sign over the given interval.")

    while error_act > error_tol and iterations < max_iter:
        fa, fb = f(a), f(b)
        x_1 = a - ((b - a) * fa) / (fb - fa)
        fx = f(x_1)

        if fx == 0:
            break
        elif fa * fx < 0:
            b = x_1
        else:
            a = x_1

        if x_1 != 0:
            error_act = abs((x_1 - x_0) / x_1)
        else:
            error_act = 0

        x_0 = x_1
        iterations += 1

    return x_0, error_act, iterations

# Hybrid method (bisection + false position)
def hybrid_bisection_false_position(a, b, f, error_tol):
    x_0 = 0
    error_act = 1
    iterations = 0

    if f(a) * f(b) > 0:
        raise ValueError("The function does not change sign over the given interval.")

    while error_act > error_tol and iterations < max_iter:
        fa, fb = f(a), f(b)
        if abs((fb - fa) / (b - a)) > 3:
            x_1 = (a + b) / 2
        else:
            x_1 = a - ((b - a) * fa) / (fb - fa)
        fx = f(x_1)

        if fx == 0:
            break
        elif fa * fx < 0:
            b = x_1
        else:
            a = x_1

        if x_1 != 0:
            error_act = abs((x_1 - x_0) / x_1)
        else:
            error_act = 0

        x_0 = x_1
        iterations += 1

    return x_0, error_act, iterations

# Roots and results
methods = {
    'Bisection': bisection,
    'False position': false_position,
    'Hybrid': hybrid_bisection_false_position
}

for method, function in methods.items():
    try:
        root, error, iterations = function(a, b, f, error_tol)
        iter_est = int(np.log2((b - a) / error_tol))
        print(f"\nResults for the {method} method:")
        print(f"Proposed root: {root}")
        print(f"Function evaluated at the root: {f(root)}")
        print(f"Error reached: {error}")
        print(f"Number of iterations: {iterations}")
        print(f"Estimated number of iterations: {iter_est}")
    except ValueError as e:
        print(f"\n{method}: {e}")

# Plot of the function and the roots
x_vals = np.linspace(a, b, 1000)
y_vals = f(x_vals)

plt.figure(figsize=(14, 8))
plt.plot(x_vals, y_vals, label='f(x)', color='black')
plt.axhline(0, color='gray', linestyle='--')

for method, function in methods.items():
    try:
        root, _, _ = function(a, b, f, error_tol)
        plt.scatter(root, 0, label=f'{method} root: {root:.6f}', s=100)
    except ValueError:
        pass

plt.xlabel('x')
plt.ylabel('f(x)')
plt.legend()
plt.grid(True)
plt.title('Approximate Roots via Bisection, False Position, and Hybrid Method')
plt.show()

# Iterations vs. target error for each method
test_tolerances = [10**-12, 10**-10, 10**-8, 10**-6, 10**-4, 10**-2]

for method, function in methods.items():
    iteration_counts = []
    for tol in test_tolerances:
        try:
            _, _, it = function(a, b, f, tol)
            iteration_counts.append(it)
        except ValueError:
            iteration_counts.append(None)

    plt.figure(figsize=(12, 8))
    plt.plot(test_tolerances, iteration_counts, marker='o', label=method)
    plt.xscale('log')
    plt.xlabel('Target error')
    plt.ylabel('Number of Iterations')
    plt.xticks(test_tolerances, labels=[f'$10^{{-{int(np.log10(e))}}}$' for e in test_tolerances])
    plt.grid(True)
    plt.legend()
    plt.title(f'Number of Iterations vs. Target Error ({method})')
    plt.show()


b)
$$
x - 2^{-x} = 0
$$


In [ ]:
# Function under study
def f(x):
    return x - 2**-x

# General parameters
error_tol = 1e-6
max_iter = 1000

# Interval [a, b]
a, b = 0.5, 1

# Bisection method
def bisection(a, b, f, error_tol):
    valid_bracket = 0
    error = 1
    iterations = 0
    x_prev = 0

    # Validate the bracket
    if f(a) * f(b) < 0:
        valid_bracket = 1
    elif f(a) * f(b) == 0:
        if f(a) == 0:
            return a, 0, 0
        if f(b) == 0:
            return b, 0, 0
        return None, None, None

    estimated_iterations = int(np.log2((b - a) / error_tol))

    while valid_bracket == 1 and error > error_tol and iterations < max_iter:
        x_new = (a + b) / 2

        if f(a) * f(x_new) < 0:
            b = x_new
        else:
            a = x_new

        # Guard against division by zero
        if x_new != 0:
            error = abs((x_new - x_prev) / x_new)
        else:
            error = 0

        x_prev = x_new
        iterations += 1

        if f(x_new) == 0:
            break

    return x_new, error, iterations

# False position method
def false_position(a, b, f, error_tol):
    x_0 = 0
    error_act = 1
    iterations = 0

    if f(a) * f(b) > 0:
        raise ValueError("The function does not change sign over the given interval.")

    while error_act > error_tol and iterations < max_iter:
        fa, fb = f(a), f(b)
        x_1 = a - ((b - a) * fa) / (fb - fa)
        fx = f(x_1)

        if fx == 0:
            break
        elif fa * fx < 0:
            b = x_1
        else:
            a = x_1

        if x_1 != 0:
            error_act = abs((x_1 - x_0) / x_1)
        else:
            error_act = 0

        x_0 = x_1
        iterations += 1

    return x_0, error_act, iterations

# Hybrid method (bisection + false position)
def hybrid_bisection_false_position(a, b, f, error_tol):
    x_0 = 0
    error_act = 1
    iterations = 0

    if f(a) * f(b) > 0:
        raise ValueError("The function does not change sign over the given interval.")

    while error_act > error_tol and iterations < max_iter:
        fa, fb = f(a), f(b)
        if abs((fb - fa) / (b - a)) > 3:
            x_1 = (a + b) / 2
        else:
            x_1 = a - ((b - a) * fa) / (fb - fa)
        fx = f(x_1)

        if fx == 0:
            break
        elif fa * fx < 0:
            b = x_1
        else:
            a = x_1

        if x_1 != 0:
            error_act = abs((x_1 - x_0) / x_1)
        else:
            error_act = 0

        x_0 = x_1
        iterations += 1

    return x_0, error_act, iterations

# Roots and results
methods = {
    'Bisection': bisection,
    'False position': false_position,
    'Hybrid': hybrid_bisection_false_position
}

for method, function in methods.items():
    try:
        root, error, iterations = function(a, b, f, error_tol)
        iter_est = int(np.log2((b - a) / error_tol))
        print(f"\nResults for the {method} method:")
        print(f"Proposed root: {root}")
        print(f"Function evaluated at the root: {f(root)}")
        print(f"Error reached: {error}")
        print(f"Number of iterations: {iterations}")
        print(f"Estimated number of iterations: {iter_est}")
    except ValueError as e:
        print(f"\n{method}: {e}")

# Plot of the function and the roots
x_vals = np.linspace(a, b, 1000)
y_vals = f(x_vals)

plt.figure(figsize=(14, 8))
plt.plot(x_vals, y_vals, label='f(x)', color='black')
plt.axhline(0, color='gray', linestyle='--')

for method, function in methods.items():
    try:
        root, _, _ = function(a, b, f, error_tol)
        plt.scatter(root, 0, label=f'{method} root: {root:.6f}', s=100)
    except ValueError:
        pass

plt.xlabel('x')
plt.ylabel('f(x)')
plt.legend()
plt.grid(True)
plt.title('Approximate Roots via Bisection, False Position, and Hybrid Method')
plt.show()

# Iterations vs. target error for each method
test_tolerances = [10**-12, 10**-10, 10**-8, 10**-6, 10**-4, 10**-2]

for method, function in methods.items():
    iteration_counts = []
    for tol in test_tolerances:
        try:
            _, _, it = function(a, b, f, tol)
            iteration_counts.append(it)
        except ValueError:
            iteration_counts.append(None)

    plt.figure(figsize=(12, 8))
    plt.plot(test_tolerances, iteration_counts, marker='o', label=method)
    plt.xscale('log')
    plt.xlabel('Target error')
    plt.ylabel('Number of Iterations')
    plt.xticks(test_tolerances, labels=[f'$10^{{-{int(np.log10(e))}}}$' for e in test_tolerances])
    plt.grid(True)
    plt.legend()
    plt.title(f'Number of Iterations vs. Target Error ({method})')
    plt.show()


c)

$$
2x \cos(x) - (x+1)^2 = 0
$$


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Function under study
def f(x):
    return 2 * x * np.cos(x) - (x + 1)**2

# General parameters
error_tol = 1e-6
max_iter = 1000

# Interval [a, b]
a, b = -2, 0

# Bisection method
def bisection(a, b, f, error_tol):
    valid_bracket = 0
    error = 1
    iterations = 0
    x_prev = 0
    x_new = (a + b) / 2  # Initialize x_new

    # Validate the bracket
    if f(a) * f(b) < 0:
        valid_bracket = 1
    elif f(a) * f(b) == 0:
        if f(a) == 0:
            return a, 0, 0
        if f(b) == 0:
            return b, 0, 0
        return None, None, None

    estimated_iterations = int(np.log2((b - a) / error_tol))

    while valid_bracket == 1 and error > error_tol and iterations < max_iter:
        x_new = (a + b) / 2

        if f(a) * f(x_new) < 0:
            b = x_new
        else:
            a = x_new

        # Guard against division by zero
        if x_new != 0:
            error = abs((x_new - x_prev) / x_new)
        else:
            error = 0

        x_prev = x_new
        iterations += 1

        if f(x_new) == 0:
            break

    # Return valid values even if the loop never ran
    return x_new, error, iterations

# False position method
def false_position(a, b, f, error_tol):
    x_0 = 0
    error_act = 1
    iterations = 0

    if f(a) * f(b) > 0:
        raise ValueError("The function does not change sign over the given interval.")

    while error_act > error_tol and iterations < max_iter:
        fa, fb = f(a), f(b)
        x_1 = a - ((b - a) * fa) / (fb - fa)
        fx = f(x_1)

        if fx == 0:
            break
        elif fa * fx < 0:
            b = x_1
        else:
            a = x_1

        if x_1 != 0:
            error_act = abs((x_1 - x_0) / x_1)
        else:
            error_act = 0

        x_0 = x_1
        iterations += 1

    return x_0, error_act, iterations

# Hybrid method (bisection + false position)
def hybrid_bisection_false_position(a, b, f, error_tol):
    x_0 = 0
    error_act = 1
    iterations = 0

    if f(a) * f(b) > 0:
        raise ValueError("The function does not change sign over the given interval.")

    while error_act > error_tol and iterations < max_iter:
        fa, fb = f(a), f(b)
        if abs((fb - fa) / (b - a)) > 3:
            x_1 = (a + b) / 2
        else:
            x_1 = a - ((b - a) * fa) / (fb - fa)
        fx = f(x_1)

        if fx == 0:
            break
        elif fa * fx < 0:
            b = x_1
        else:
            a = x_1

        if x_1 != 0:
            error_act = abs((x_1 - x_0) / x_1)
        else:
            error_act = 0

        x_0 = x_1
        iterations += 1

    return x_0, error_act, iterations

# Roots and results
methods = {
    'Bisection': bisection,
    'False position': false_position,
    'Hybrid': hybrid_bisection_false_position
}

for method, function in methods.items():
    try:
        root, error, iterations = function(a, b, f, error_tol)
        iter_est = int(np.log2((b - a) / error_tol))
        print(f"\nResults for the {method} method:")
        print(f"Proposed root: {root}")
        print(f"Function evaluated at the root: {f(root)}")
        print(f"Error reached: {error}")
        print(f"Number of iterations: {iterations}")
        print(f"Estimated number of iterations: {iter_est}")
    except ValueError as e:
        print(f"\n{method}: {e}")

# Plot of the function and the roots
x_vals = np.linspace(a, b, 1000)
y_vals = f(x_vals)

plt.figure(figsize=(14, 8))
plt.plot(x_vals, y_vals, label='f(x)', color='black')
plt.axhline(0, color='gray', linestyle='--')

for method, function in methods.items():
    try:
        root, _, _ = function(a, b, f, error_tol)
        plt.scatter(root, 0, label=f'{method} root: {root:.6f}', s=100)
    except ValueError:
        pass

plt.xlabel('x')
plt.ylabel('f(x)')
plt.legend()
plt.grid(True)
plt.title('Approximate Roots via Bisection, False Position, and Hybrid Method')
plt.show()

# Iterations vs. target error for each method
test_tolerances = [10**-12, 10**-10, 10**-8, 10**-6, 10**-4, 10**-2]

for method, function in methods.items():
    iteration_counts = []
    for tol in test_tolerances:
        try:
            _, _, it = function(a, b, f, tol)
            iteration_counts.append(it)
        except ValueError:
            iteration_counts.append(None)

    plt.figure(figsize=(12, 8))
    plt.plot(test_tolerances, iteration_counts, marker='o', label=method)
    plt.xscale('log')
    plt.xlabel('Target error')
    plt.ylabel('Number of Iterations')
    plt.xticks(test_tolerances, labels=[f'$10^{{-{int(np.log10(e))}}}$' for e in test_tolerances])
    plt.grid(True)
    plt.legend()
    plt.title(f'Number of Iterations vs. Target Error ({method})')
    plt.show()


d)
$$
e^x - 2 = \cos\left(e^x - 2\right)
$$


In [ ]:
# Function under study
def f(x):
    return np.exp(x) - 2 - np.cos(np.exp(x)-2)

# General parameters
error_tol = 1e-6
max_iter = 1000

# Interval [a, b]
a, b = 0, 2

# Bisection method
def bisection(a, b, f, error_tol):
    valid_bracket = 0
    error = 1
    iterations = 0
    x_prev = 0

    # Validate the bracket
    if f(a) * f(b) < 0:
        valid_bracket = 1
    elif f(a) * f(b) == 0:
        if f(a) == 0:
            return a, 0, 0
        if f(b) == 0:
            return b, 0, 0
        return None, None, None

    estimated_iterations = int(np.log2((b - a) / error_tol))

    while valid_bracket == 1 and error > error_tol and iterations < max_iter:
        x_new = (a + b) / 2

        if f(a) * f(x_new) < 0:
            b = x_new
        else:
            a = x_new

        # Guard against division by zero
        if x_new != 0:
            error = abs((x_new - x_prev) / x_new)
        else:
            error = 0

        x_prev = x_new
        iterations += 1

        if f(x_new) == 0:
            break

    return x_new, error, iterations

# False position method
def false_position(a, b, f, error_tol):
    x_0 = 0
    error_act = 1
    iterations = 0

    if f(a) * f(b) > 0:
        raise ValueError("The function does not change sign over the given interval.")

    while error_act > error_tol and iterations < max_iter:
        fa, fb = f(a), f(b)
        x_1 = a - ((b - a) * fa) / (fb - fa)
        fx = f(x_1)

        if fx == 0:
            break
        elif fa * fx < 0:
            b = x_1
        else:
            a = x_1

        if x_1 != 0:
            error_act = abs((x_1 - x_0) / x_1)
        else:
            error_act = 0

        x_0 = x_1
        iterations += 1

    return x_0, error_act, iterations

# Hybrid method (bisection + false position)
def hybrid_bisection_false_position(a, b, f, error_tol):
    x_0 = 0
    error_act = 1
    iterations = 0

    if f(a) * f(b) > 0:
        raise ValueError("The function does not change sign over the given interval.")

    while error_act > error_tol and iterations < max_iter:
        fa, fb = f(a), f(b)
        if abs((fb - fa) / (b - a)) > 3:
            x_1 = (a + b) / 2
        else:
            x_1 = a - ((b - a) * fa) / (fb - fa)
        fx = f(x_1)

        if fx == 0:
            break
        elif fa * fx < 0:
            b = x_1
        else:
            a = x_1

        if x_1 != 0:
            error_act = abs((x_1 - x_0) / x_1)
        else:
            error_act = 0

        x_0 = x_1
        iterations += 1

    return x_0, error_act, iterations

# Roots and results
methods = {
    'Bisection': bisection,
    'False position': false_position,
    'Hybrid': hybrid_bisection_false_position
}

for method, function in methods.items():
    try:
        root, error, iterations = function(a, b, f, error_tol)
        iter_est = int(np.log2((b - a) / error_tol))
        print(f"\nResults for the {method} method:")
        print(f"Proposed root: {root}")
        print(f"Function evaluated at the root: {f(root)}")
        print(f"Error reached: {error}")
        print(f"Number of iterations: {iterations}")
        print(f"Estimated number of iterations: {iter_est}")
    except ValueError as e:
        print(f"\n{method}: {e}")

# Plot of the function and the roots
x_vals = np.linspace(a, b, 1000)
y_vals = f(x_vals)

plt.figure(figsize=(14, 8))
plt.plot(x_vals, y_vals, label='f(x)', color='black')
plt.axhline(0, color='gray', linestyle='--')

for method, function in methods.items():
    try:
        root, _, _ = function(a, b, f, error_tol)
        plt.scatter(root, 0, label=f'{method} root: {root:.6f}', s=100)
    except ValueError:
        pass

plt.xlabel('x')
plt.ylabel('f(x)')
plt.legend()
plt.grid(True)
plt.title('Approximate Roots via Bisection, False Position, and Hybrid Method')
plt.show()

# Iterations vs. target error for each method
test_tolerances = [10**-12, 10**-10, 10**-8, 10**-6, 10**-4, 10**-2]

for method, function in methods.items():
    iteration_counts = []
    for tol in test_tolerances:
        try:
            _, _, it = function(a, b, f, tol)
            iteration_counts.append(it)
        except ValueError:
            iteration_counts.append(None)

    plt.figure(figsize=(12, 8))
    plt.plot(test_tolerances, iteration_counts, marker='o', label=method)
    plt.xscale('log')
    plt.xlabel('Target error')
    plt.ylabel('Number of Iterations')
    plt.xticks(test_tolerances, labels=[f'$10^{{-{int(np.log10(e))}}}$' for e in test_tolerances])
    plt.grid(True)
    plt.legend()
    plt.title(f'Number of Iterations vs. Target Error ({method})')
    plt.show()
